# Week 3 Day 2 — Match Winner & Top Player Models

Uses Day-1 feature tables and the same time split (`year < 2024` train, `>= 2024` holdout).

**Artifacts:** `models/*.joblib` · **API:** `predict.py`


## Setup

In [ ]:
from pathlib import Path
import sys, json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

from src.config import MATCH_FEATURES, PLAYER_FEATURES, RESULTS_DIR, HOLDOUT_YEAR
from src.splits import time_based_split, describe_split
from src.baselines import (
    match_baseline_always_home_with_prior,
    match_baseline_higher_ladder,
    player_baseline_prev_leader,
)
from src.match_model import train_match_models, match_feature_importance
from src.player_model import train_player_model
from src.metrics import topk_hit_rate, regression_report_dict

sns.set_theme(style="whitegrid")
print("holdout year", HOLDOUT_YEAR)


## Task 1 — Baselines

**Match:** always-home (with train prior probability) and higher pre-match ladder %.

**Player:** rank by recent form (`avg_impact_l3` / `avg_impact_l5`).


In [ ]:
matches = pd.read_csv(MATCH_FEATURES, parse_dates=["match_date"])
train_m, hold_m = time_based_split(matches, holdout_year=HOLDOUT_YEAR)
print(describe_split(matches))

train_home_rate = float(train_m.home_win.mean())
base_home = match_baseline_always_home_with_prior(hold_m, train_home_rate)
base_ladder = match_baseline_higher_ladder(hold_m)
pd.DataFrame([{"model": "always_home", **base_home}, {"model": "higher_ladder", **base_ladder}])


In [ ]:
players = pd.read_csv(PLAYER_FEATURES, parse_dates=["match_date"])
train_p, hold_p = time_based_split(players, holdout_year=HOLDOUT_YEAR)
print(describe_split(players))
base_p = player_baseline_prev_leader(train_p, hold_p, k=5)
base_p


## Task 2 — Match winner models

Sklearn `ColumnTransformer` + **Logistic Regression** and **Gradient Boosting**.
Final pick = best holdout ROC AUC (then lower Brier).


In [ ]:
fitted, metrics, num_cols, cat_cols = train_match_models(train_m, hold_m)
match_tbl = pd.DataFrame(metrics).T
match_tbl


In [ ]:
final_name = match_tbl["roc_auc"].idxmax()
final = fitted[final_name]
print("final model:", final_name)
imp = match_feature_importance(final, num_cols, cat_cols).head(15)
display(imp)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=imp, y="feature", x="abs_value", ax=ax, color="steelblue")
ax.set_title(f"Top features ({final_name})")
fig.tight_layout()
plt.show()


### Why GBM as final?
On this holdout, GBM beats always-home and is a bit better than the ladder baseline on accuracy, with stronger ROC AUC and Brier. Logreg is still useful to read coefficients.


## Task 3 — Top player model

Regress `impact_score` from pre-match features, then rank players in the match.
A full ranking model would be heavier; this already gives a sorted list.

Score used in the API: `0.3 * model + 0.7 * avg_impact_l5` (helps top-k a bit).


In [ ]:
pipe, pmetrics, p_num, p_cat, scored = train_player_model(train_p, hold_p, k=5)
w = 0.3
scored["pred_blend"] = w * scored["pred_impact"] + (1 - w) * scored["avg_impact_l5"].fillna(0)
blend = {
    **regression_report_dict(scored["impact_score"], scored["pred_blend"]),
    "topk5_hit_rate": topk_hit_rate(
        scored, game_col="game_key", pred_col="pred_blend", actual_col="impact_score", k=5
    ),
}
print("raw model", pmetrics)
print("blend", blend)
print("baseline", base_p)


## Task 4 — Sanity / sniff tests

Early holdout games often show flat ladder % (0.5), so form margins matter more. When the model disagrees with a simple home tip, it is usually following L5/L8 form.


In [ ]:
# Prefer saved sniff file from train_models.py
sniff_path = RESULTS_DIR / "sniff_tests.csv"
if sniff_path.exists():
    sniff = pd.read_csv(sniff_path)
else:
    sniff = hold_m.head(3)
display(sniff)

metrics_path = RESULTS_DIR / "metrics.json"
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2)[:1200])


In [ ]:
p_imp = pd.read_csv(RESULTS_DIR / "player_feature_importance.csv").head(12)
display(p_imp)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=p_imp, y="feature", x="value", ax=ax, color="darkorange")
ax.set_title("Player model importance proxy (GBM sample)")
fig.tight_layout()
plt.show()


### Leakage check
Top match features are form, ladder, and H2H diffs. Same-game scores/margins are not used as inputs. Player importances are mostly prior impact/disposals averages.


## Task 5 — Callable API

```python
from predict import predict_match_winner, predict_top_player

predict_match_winner("Geelong Cats", "Richmond Tigers", "2024-06-01", home_team="Geelong Cats")
predict_top_player(team="Geelong Cats", match_date="2024-06-01", top_k=5)
```

Rebuild artifacts: `python train_models.py`


In [ ]:
from predict import predict_match_winner, predict_top_player

row = hold_m.iloc[0]
print(predict_match_winner(row.home_team, row.away_team, str(pd.Timestamp(row.match_date).date()), home_team=row.home_team))
print(predict_top_player(team=row.home_team, match_date=str(pd.Timestamp(row.match_date).date()), opponent=row.away_team, top_k=5))
